# Nepal E-Commerce Sales Intelligence
## Data Cleaning & Preparation

---

**Business Analytics Portfolio Project**

| | |
|---|---|
| **Author** | Sakshyam Pandit |
| **Date** | September 2026 |
| **Tools** | Python, Pandas, NumPy, Matplotlib, Seaborn |
| **Dataset** | Nepal E-Commerce Transactions (Synthetic) |

---

## Executive Summary

This notebook handles the data cleaning and preparation phase of our Nepal E-Commerce analysis. The raw dataset contains **1,215 e-commerce transactions** across six major cities in Nepal, with intentional data quality issues that mirror real-world scenarios.

**Key Cleaning Actions:**
- Removed 15 duplicate records
- Imputed missing values in Rating (3.95%), CustomerAge (4.28%), and PaymentMethod (3.95%)
- Capped 168 price outliers using IQR method
- Engineered 6 new features for downstream analysis

**Business Impact:** Clean, reliable data ensures accurate insights for revenue optimization, customer segmentation, and inventory planning decisions.

---
## 1. Setup & Configuration

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from style_config import setup_professional_style, format_currency, format_number
from utils import load_and_generate_data, clean_data, engineer_features

setup_professional_style()

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print('Environment configured successfully.')

---
## 2. Data Loading

The dataset represents e-commerce transactions from an online store operating across Nepal. In production, this would load from `data/raw/sales.csv`.

In [ ]:
df_raw = load_and_generate_data(n_rows=1200, random_state=42)
print(f'Raw dataset loaded: {df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns')
df_raw.head(10)

---
## 3. Initial Data Inspection

Before cleaning, we must understand the data structure and identify quality issues.

### 3.1 Data Structure

In [ ]:
print('Dataset Shape:', df_raw.shape)
print('\nColumn Data Types:')
df_raw.info()

### 3.2 Statistical Summary

In [ ]:
df_raw.describe(include='all').T

---
## 4. Data Quality Assessment

**Business Question:** *What is the quality of our data, and what issues need to be addressed before analysis?*

Data quality directly impacts the reliability of our business insights. We assess three key dimensions:
1. **Missing Values** - Incomplete records
2. **Duplicates** - Redundant entries
3. **Outliers** - Anomalous values

### 4.1 Missing Values Analysis

In [ ]:
missing_summary = df_raw.isna().sum().to_frame('missing_count')
missing_summary['missing_%'] = np.round(100 * missing_summary['missing_count'] / len(df_raw), 2)
missing_summary = missing_summary[missing_summary['missing_count'] > 0].sort_values('missing_%', ascending=False)

print('Missing Values Summary:')
print('=' * 40)
missing_summary

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(df_raw.isna(), cbar=False, yticklabels=False, cmap='rocket', ax=ax)
ax.set_title('Missing Value Map - Raw Data', fontsize=14, fontweight='bold')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

**Business Insight:** Missing data affects ~4% of records across Rating, CustomerAge, and PaymentMethod. This level is manageable with imputation and won't significantly impact analysis accuracy.

### 4.2 Duplicate Analysis

In [ ]:
dup_count = df_raw.duplicated().sum()
dup_pct = 100 * dup_count / len(df_raw)

print(f'Fully duplicated rows: {dup_count} ({dup_pct:.2f}%)')
print(f'Unique rows: {len(df_raw) - dup_count:,}')

### 4.3 Outlier Detection - Price

In [ ]:
Q1, Q3 = df_raw['Price'].quantile([0.25, 0.75])
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outlier_count = ((df_raw['Price'] < lower_bound) | (df_raw['Price'] > upper_bound)).sum()

print(f'Price Distribution:')
print(f'  Q1: {format_currency(Q1)}')
print(f'  Q3: {format_currency(Q3)}')
print(f'  IQR: {format_currency(IQR)}')
print(f'  Lower Bound: {format_currency(lower_bound)}')
print(f'  Upper Bound: {format_currency(upper_bound)}')
print(f'  Outliers Detected: {outlier_count}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sns.boxplot(x=df_raw['Price'], color='#e53e3e', ax=ax, width=0.5)
ax.set_title('Price Distribution - Before Cleaning', fontsize=14, fontweight='bold')
ax.set_xlabel('Price (NPR)')
plt.tight_layout()
plt.show()

---
## 5. Data Cleaning

**Business Question:** *How do we transform raw data into analysis-ready format while preserving data integrity?*

Our cleaning strategy:
1. **Remove duplicates** - Eliminate redundant records
2. **Impute missing values** - Use median (robust to outliers) for numeric, mode for categorical
3. **Cap outliers** - IQR method preserves rows while limiting extreme values

In [ ]:
df_clean = clean_data(df_raw)

print('Cleaning Summary:')
print('=' * 40)
print(f'Rows before: {len(df_raw):,}')
print(f'Rows after:  {len(df_clean):,}')
print(f'Rows removed: {len(df_raw) - len(df_clean):,}')
print(f'\nMissing values remaining: {df_clean.isna().sum().sum()}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(x=df_raw['Price'], color='#e53e3e', ax=axes[0], width=0.5)
axes[0].set_title('Price - Before Cleaning', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Price (NPR)')

sns.boxplot(x=df_clean['Price'], color='#38a169', ax=axes[1], width=0.5)
axes[1].set_title('Price - After Cleaning', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Price (NPR)')

plt.suptitle('Price Distribution Comparison', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 6. Feature Engineering

**Business Question:** *What derived features can we create to enable deeper analysis?*

Feature engineering transforms raw data into meaningful business metrics:
- **TotalAmount** - Revenue per transaction
- **Month/MonthNum** - Temporal analysis
- **Weekday** - Day-of-week patterns
- **AgeGroup** - Customer segmentation
- **RevenueSegment** - Customer value tiers

In [ ]:
df_clean = engineer_features(df_clean)

print('New Features Created:')
print('=' * 40)
print(f'TotalAmount: {format_currency(df_clean["TotalAmount"].sum())}')
print(f'Months covered: {df_clean["Month"].nunique()}')
print(f'Age Groups: {df_clean["AgeGroup"].unique().tolist()}')
print(f'Revenue Segments: {df_clean["RevenueSegment"].unique().tolist()}')

df_clean.head(10)

---
## 7. Cleaned Data Validation

In [ ]:
print('Cleaned Dataset Validation:')
print('=' * 50)
print(f'Shape: {df_clean.shape}')
print(f'Missing values: {df_clean.isna().sum().sum()}')
print(f'Duplicates: {df_clean.duplicated().sum()}')
print(f'\nColumn dtypes:')
print(df_clean.dtypes)
print(f'\nMemory usage: {df_clean.memory_usage(deep=True).sum() / 1024:.2f} KB')

In [ ]:
df_clean.describe().T

---
## 8. Save Processed Data

In [ ]:
df_clean.to_csv('../data/processed/cleaned_sales.csv', index=False)

print('Cleaned data saved to:')
print('  - data/processed/cleaned_sales.csv')

---
## 9. Summary & Business Insights

| Metric | Value |
|--------|-------|
| Raw Records | 1,215 |
| Clean Records | 1,200 |
| Duplicates Removed | 15 |
| Outliers Capped | 168 |
| Missing Values Filled | 148 |
| New Features Created | 6 |

**Key Takeaways:**
1. Data quality issues were minimal (~4% missing, ~1.2% duplicates)
2. IQR capping preserved all records while handling extreme price values
3. Feature engineering created meaningful business metrics for downstream analysis
4. Cleaned data is ready for exploratory analysis and predictive modeling

**Next Step:** Proceed to `02_exploratory_analysis.ipynb` for KPI dashboard and deep-dive analysis.